<h1>Importing Libraries</h1>

In [ ]:
import math
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import GradientBoostingClassifier
import xgboost as xgb
import lightgbm as lgb

import graphviz

from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import KFold
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_predict
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, roc_curve, roc_auc_score
from sklearn.feature_selection import mutual_info_classif
from sklearn.preprocessing import LabelEncoder
from imblearn.over_sampling import RandomOverSampler
from collections import Counter

<h1>Importing The Dataset</h1>

In [ ]:
lung_cancer = pd.read_csv('lung_cancer.csv')
lung_cancer

In [ ]:
lung_cancer.isnull().sum()
le = LabelEncoder()

In [ ]:
lung_cancer.GENDER = le.fit_transform(lung_cancer['GENDER'])
lung_cancer.LUNG_CANCER = le.fit_transform(lung_cancer['LUNG_CANCER'])

In [ ]:
x = lung_cancer.drop(['LUNG_CANCER'], axis = 1)
y = lung_cancer['LUNG_CANCER']
mutual_info = mutual_info_classif(x,y)
mutual_info = pd.Series(mutual_info)
mutual_info.index = x.columns
mutual_info.sort_values(ascending=False)

In [ ]:
lung_cancer.mean()
lung_cancer.min()
lung_cancer.max()
lung_cancer.std()

<h1>Visualizing The Data</h1>

In [ ]:
plt.figure(figsize=(6,6))
names = ["Yes", "No"]
count = [(lung_cancer.LUNG_CANCER.values == 1).sum(), (lung_cancer.LUNG_CANCER.values == 0).sum()]
plt.bar(names, count, color = ["Red", "Green"])
plt.title('Lung Cancer', color = 'blue', fontsize= 25)
plt.xlabel('Lung Cancer', fontsize= 18)
plt.ylabel('Count', fontsize= 18)
plt.ylim(0,290)
for i in range(len(names)):
    plt.text(i, count[i], count[i], ha='center', va='bottom', fontsize=16)
plt.show()

In [ ]:
plt.figure(figsize = (20, 10))
sns.heatmap(lung_cancer.corr(), annot = True, cmap="plasma")

In [ ]:
os = RandomOverSampler(0.7)
x_os, y_os = os.fit_resample(x,y)
print("Before fit {}".format(Counter(y)))
print("After fit {}".format(Counter(y_os)))

In [ ]:
plt.figure(figsize=(6,6))
names = ["Yes", "No"]
count = [270, 216]
plt.bar(names, count, color = ["Red", "Green"])
plt.title('Lung Cancer', color = 'blue', fontsize= 25)
plt.xlabel('Lung Cancer', fontsize= 18)
plt.ylabel('Count', fontsize= 18)
plt.ylim(0,290)
for i in range(len(names)):
    plt.text(i, count[i], count[i], ha='center', va='bottom', fontsize=16)
plt.show()

In [ ]:
scaler = StandardScaler()
scaler.fit(x_os)
scaled_x = scaler.transform(x_os)

In [ ]:
pca = PCA(n_components=7)
pca.fit(scaled_x)
pca_x = pca.transform(scaled_x)
pca_x.shape

<h1>Models Training</h1>

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(pca_x,y_os,test_size = 0.35)

GBM = GradientBoostingClassifier(n_estimators=100, learning_rate=1.0, max_depth=1, random_state=45).fit(x_train, y_train)

XGB = xgb.XGBClassifier(objective="binary:logistic", random_state=45, eval_metric="auc", n_estimators=50).fit(x_train, y_train)

LGBM = lgb.LGBMClassifier(num_leaves=31, learning_rate=1, n_estimators=100, random_state=45).fit(x_train, y_train)

In [ ]:
GBM_pre = GBM.predict(x_test)
GBM_sc = GBM.score(x_test, y_test) * 100
GBM_sc = "{:.2f}".format(GBM_sc)

XGB_pre = XGB.predict(x_test)
XGB_sc = XGB.score(x_test, y_test) * 100
XGB_sc = "{:.2f}".format(XGB_sc)

LGBM_pre = LGBM.predict(x_test)
LGBM_sc = LGBM.score(x_test, y_test) * 100
LGBM_sc = "{:.2f}".format(LGBM_sc)

In [ ]:
algo = ['GBM', 'XGB', 'LGBM']
acc = [math.floor(int(float(GBM_sc))),math.floor(int(float(XGB_sc))), math.floor(int(float(LGBM_sc)))]
pert = [GBM_sc, XGB_sc, LGBM_sc]
colors = ['blue','magenta', 'green']
plt.bar(algo, acc,color=colors, edgecolor=colors)
plt.title('Accuracy', color = 'coral', fontsize= 23)
plt.xlabel('Classifier Name', fontsize= 18)
plt.ylabel('Accuracy Percentage', fontsize= 18)
plt.ylim(70,102)
for i in range(len(algo)):
    plt.text(i, acc[i], pert[i], ha='center', va='bottom', fontsize=16)
plt.show()

In [ ]:
def print_confusion_matrix(confusion_matrix, class_names, figsize = (7,4), fontsize=14, title = "Confusion Matrix"):
    df_cm = pd.DataFrame(
        confusion_matrix, index=class_names, columns=class_names, 
    )
    fig = plt.figure(figsize=figsize)
    try:
        heatmap = sns.heatmap(df_cm, annot=True, fmt="d")
    except ValueError:
        raise ValueError("Confusion matrix values must be integers.")
    heatmap.yaxis.set_ticklabels(heatmap.yaxis.get_ticklabels(), rotation=90, ha='right', fontsize=fontsize)
    heatmap.xaxis.set_ticklabels(heatmap.xaxis.get_ticklabels(), rotation=0, ha='center', fontsize=fontsize)
    plt.ylabel('Actual', fontsize=18, color='blue')
    plt.xlabel('Prediction',  fontsize=18, color='blue')
    plt.title(title, fontsize=22, color='blue')

In [ ]:
cm = confusion_matrix(y_test, GBM_pre)
print_confusion_matrix(cm,["0", "1"], title="GradientBoostingClassifier (GBM)")

In [ ]:
cm = confusion_matrix(y_test, XGB_pre)
print_confusion_matrix(cm,["0", "1"], title="XGBoost (XGB)")

In [ ]:
cm = confusion_matrix(y_test, LGBM_pre)
print_confusion_matrix(cm,["0", "1"], title="LightGradientBoostingClassifier (LGBM)")

In [ ]:
pd.DataFrame(classification_report(y_test, GBM_pre, output_dict=True))

In [ ]:
pd.DataFrame(classification_report(y_test, XGB_pre, output_dict=True))

In [ ]:
pd.DataFrame(classification_report(y_test, LGBM_pre, output_dict=True))

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(x_os,y_os,test_size = 0.35)

GBM = GradientBoostingClassifier(n_estimators=100, learning_rate=1.0, max_depth=1, random_state=45).fit(x_train, y_train)

XGB = xgb.XGBClassifier(objective="binary:logistic", random_state=45, eval_metric="auc", n_estimators=50).fit(x_train, y_train)

LGBM = lgb.LGBMClassifier(num_leaves=31, learning_rate=1, n_estimators=100, random_state=45).fit(x_train, y_train)

In [ ]:
plt.figure(figsize=(10,9))
xgb.plot_importance(XGB)
plt.show()

In [ ]:
import shap
explainer = shap.Explainer(GBM, x_train)
shap_values = explainer(x_train)
shap.plots.waterfall(shap_values[0])

In [ ]:
shap.plots.bar(shap_values)

In [ ]:
shap.plots.beeswarm(shap_values)

In [ ]:
explainer = shap.Explainer(XGB, x_train)
shap_values = explainer(x_train)

In [ ]:
shap.plots.waterfall(shap_values[0])

In [ ]:
shap.plots.bar(shap_values)

In [ ]:
shap.plots.beeswarm(shap_values)

In [ ]:
explainer = shap.Explainer(LGBM, x_train)
shap_values = explainer(x_train)

In [ ]:
shap.plots.waterfall(shap_values[0])

In [ ]:
shap.plots.bar(shap_values)

In [ ]:
shap.plots.beeswarm(shap_values)

In [ ]:
GBM_probs_tr = GBM.predict_proba(x_train)[:,1]
auc_tr = roc_auc_score(y_train, GBM_probs_tr)
fpr_tr, tpr_tr, thresholds = roc_curve(y_train, GBM_probs_tr)

GBM_probs_ts = GBM.predict_proba(x_test)[:,1]
auc_ts = roc_auc_score(y_test, GBM_probs_ts)
fpr_ts, tpr_ts, thresholds = roc_curve(y_test, GBM_probs_ts)

plt.plot(fpr_tr,tpr_tr, color='purple', label="T rain AUC : {:.3f}".format(auc_tr))
plt.plot(fpr_ts,tpr_ts, color='green', label="T est AUC : {:.3f}".format(auc_ts))
plt.plot([0,1], [0,1], color='darkblue', linestyle='--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('GBM ROC Curve')
plt.legend()
plt.show()

In [ ]:
XGB_probs_tr = XGB.predict_proba(x_train)[:,1]
auc_tr = roc_auc_score(y_train, XGB_probs_tr)
fpr_tr, tpr_tr, thresholds = roc_curve(y_train, XGB_probs_tr)

XGB_probs_ts = XGB.predict_proba(x_test)[:,1]
auc_ts = roc_auc_score(y_test, XGB_probs_ts)
fpr_ts, tpr_ts, thresholds = roc_curve(y_test, XGB_probs_ts)

plt.plot(fpr_tr,tpr_tr, color='purple', label="T rain AUC : {:.3f}".format(auc_tr))
plt.plot(fpr_ts,tpr_ts, color='green', label="T est AUC : {:.3f}".format(auc_ts))
plt.plot([0,1], [0,1], color='darkblue', linestyle='--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('XGB ROC Curve')
plt.legend()
plt.show()

In [ ]:
LGBM_probs_tr = LGBM.predict_proba(x_train)[:,1]
auc_tr = roc_auc_score(y_train, LGBM_probs_tr)
fpr_tr, tpr_tr, thresholds = roc_curve(y_train, LGBM_probs_tr)

LGBM_probs_ts = LGBM.predict_proba(x_test)[:,1]
auc_ts = roc_auc_score(y_test, LGBM_probs_ts)
fpr_ts, tpr_ts, thresholds = roc_curve(y_test, LGBM_probs_ts)

plt.plot(fpr_tr,tpr_tr, color='purple', label="T rain AUC : {:.3f}".format(auc_tr))
plt.plot(fpr_ts,tpr_ts, color='green', label="T est AUC : {:.3f}".format(auc_ts))
plt.plot([0,1], [0,1], color='darkblue', linestyle='--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('LGBM ROC Curve')
plt.legend()
plt.show()